# Part 1: Let the student write one response

The **student** is the small model we want to improve. First, it answers a question using its current behavior. That response is called a **trajectory**. We keep the exact token IDs it produced so we can inspect them in later parts.

This notebook follows [*On-Policy Distillation of Language Models: Learning from Self-Generated Mistakes*](https://arxiv.org/abs/2306.13649). The paper calls its method **Generalized Knowledge Distillation (GKD)**: the student generates a response and the teacher gives feedback on that response. We are building and checking those pieces one at a time. We have not implemented the paper's training method yet.

Run the notebook from top to bottom in Google Colab. Choose **Runtime → Change runtime type → GPU** first. We use one question at a time, so the first dimension of each token tensor is `1`. Saved outputs from earlier runs do not prove the new teacher cells ran; rerun the cells to check the updated code.

## 1. Install the notebook dependencies

This cell installs the Python packages used in the notebook. It uses `uv` for the packages; if `uv` is missing, it installs `uv` first. `sys.executable` makes sure installation goes into this notebook's Python environment. See the [uv environment guide](https://docs.astral.sh/uv/pip/environments/).

We use Transformers version 4.51 or newer for the Qwen3 student. If Colab asks you to restart the runtime after installation, do that and then start again at the top.

Some packages are for later steps. Installing them now does not load a model or start training. See the [student model's requirements](https://huggingface.co/Qwen/Qwen3-0.6B-Base#requirements).


In [1]:
import importlib.util
import subprocess
import sys

# sys.executable is the Python interpreter running this notebook's kernel.
# Bootstrap uv only if it is missing; pip is used solely to install uv itself.
if importlib.util.find_spec("uv") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "uv"],
        check=True,
    )

# uv installs into this kernel's environment, avoiding an unrelated local .venv.
# torch provides GPU tensors; transformers provides tokenization and generation.
# accelerate supports GPU placement; the remaining packages are for later milestones.
# No model is loaded here: these dependencies prepare us to sample a student trajectory.
# check=True stops this cell if installation fails, so the failure is visible.
subprocess.run(
    [
        sys.executable, "-m", "uv", "pip", "install",
        "--python", sys.executable,
        "torch", "transformers>=4.51.0,<5", "accelerate",
        "bitsandbytes", "sentencepiece", "pandas",
    ],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'uv', 'pip', 'install', '--python', '/usr/bin/python3', 'torch', 'transformers>=4.51.0,<5', 'accelerate', 'bitsandbytes', 'sentencepiece', 'pandas'], returncode=0)

## 2. Check the GPU and choose a numeric precision

The model runs on a GPU. This cell prints the GPU's name and **GPU memory**, which is separate from Colab's system RAM. It uses BF16 when available and FP16 otherwise. These formats store model weights more compactly; token IDs are still whole numbers.

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Stop early with an actionable message if Colab is still using a CPU runtime.
assert torch.cuda.is_available(), (
    "CUDA is required. In Colab, select Runtime > Change runtime type > GPU."
)

# Use the first GPU for the student and its input tokens.
device = torch.device("cuda:0")
gpu_properties = torch.cuda.get_device_properties(device)
gpu_memory_gib = gpu_properties.total_memory / (1024 ** 3)
bf16_supported = torch.cuda.is_bf16_supported()

# BF16 handles a wider range of values than FP16, so prefer it if available.
# This choice changes model weights, not integer token IDs.
model_dtype = torch.bfloat16 if bf16_supported else torch.float16

print(f"GPU name: {gpu_properties.name}")
print(f"GPU memory (total): {gpu_memory_gib:.2f} GiB")
print(f"BF16 supported: {bf16_supported}")
print(f"Selected model dtype: {model_dtype}")

GPU name: NVIDIA A100-SXM4-80GB
GPU memory (total): 79.25 GiB
BF16 supported: True
Selected model dtype: torch.bfloat16


## 3. Load the tokenizer and the student

A **tokenizer** turns text into token IDs, which are numbers the model reads. We load the tokenizer and model from the **same student checkpoint**, `Qwen/Qwen3-0.6B-Base`. This makes the token IDs consistent with the student. Our new planned teacher is `Qwen/Qwen3-32B`; we load its weights only after the compatibility check in Part 3.

`eval()` puts the student in evaluation mode, which turns off training-only behavior. It does not stop us from sampling a response.

In [3]:
# This checkpoint provides both the student's tokenizer and its weights.
# Using its own tokenizer gives the student token IDs it understands.
student_model_id = "Qwen/Qwen3-0.6B-Base"
# Record the Qwen3 teacher choice; its weights load only in Part 4.
teacher_model_id = "Qwen/Qwen3-32B"
tokenizer = AutoTokenizer.from_pretrained(student_model_id)

# Padding fills empty space when examples have different lengths. We have
# only one prompt, but generation still needs a padding token ID.
# If none exists, use the end-of-sequence token for that purpose.
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the student's weights onto GPU 0 in the BF16 or FP16 format above.
student_model = AutoModelForCausalLM.from_pretrained(
    student_model_id,
    torch_dtype=model_dtype,
    device_map={"": device.index},
)
student_model.eval()

# The tokenizer's IDs must fit within the student's output vocabulary.
assert len(tokenizer) <= student_model.config.vocab_size
print("Student tokenizer and model:", student_model_id)
print("Tokenizer size:", len(tokenizer))
print("Student vocabulary capacity:", student_model.config.vocab_size)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Student tokenizer and model: Qwen/Qwen3-0.6B-Base
Tokenizer size: 151669
Student vocabulary capacity: 151936


## 4. Tokenize and inspect the prompt

The **prompt** is the question we give the student. We use plain text here. One token might be a word, part of a word, punctuation, or a space.

`prompt_inputs` contains the token IDs and an `attention_mask`. Both have shape `[1, prompt_length]`: one question, with `prompt_length` tokens. The mask marks real input positions. Every token in this cell belongs to the prompt; the student has not answered yet.

In [4]:
# This question is the prompt the student will answer.
prompt_text = (
    "A store has 24 boxes. Each box contains 15 pencils. The store sells 87 pencils. "
    "How many pencils remain? Explain your reasoning step by step."
)

# Turn text into PyTorch tensors on the student's GPU.
# Do not add wrapper tokens: we want to inspect the plain question.
# With one prompt and no padding, attention_mask is all ones.
prompt_inputs = tokenizer(
    prompt_text,
    return_tensors="pt",
    add_special_tokens=False,
).to(device)

# input_ids has shape [1, number_of_prompt_tokens].
# Save these IDs so we can later check that generation kept the prompt.
prompt_token_ids = prompt_inputs["input_ids"]
prompt_length = prompt_token_ids.shape[1]

print("Prompt:", prompt_text)
print("Prompt token IDs:", prompt_token_ids[0].tolist())
print("input_ids tensor shape:", tuple(prompt_token_ids.shape))
print("Number of prompt tokens:", prompt_length)

Prompt: A store has 24 boxes. Each box contains 15 pencils. The store sells 87 pencils. How many pencils remain? Explain your reasoning step by step.
Prompt token IDs: [32, 3553, 702, 220, 17, 19, 14697, 13, 8886, 3745, 5610, 220, 16, 20, 96338, 13, 576, 3553, 30778, 220, 23, 22, 96338, 13, 2585, 1657, 96338, 7146, 30, 81917, 697, 32711, 3019, 553, 3019, 13]
input_ids tensor shape: (1, 36)
Number of prompt tokens: 36


## 5. Sample one response from the student

The student now writes one response. We use **sampling**, so it chooses each next token from its own possible next tokens. The response tokens are its trajectory. Because the student produced them itself, this step is **on-policy**.

We allow at most 128 new response tokens. A special end token can stop the response sooner. The fixed random seed helps reproduce this sample when the hardware and software are the same.

We turn off top-k and top-p filtering so sampling considers all possible next tokens. The returned tensor contains the **prompt first, then the response**. See the [Hugging Face generation settings](https://huggingface.co/docs/transformers/main_classes/text_generation).

In [5]:
# A new response needs new scoring checks, so clear the old success flag.
student_scoring_verified = False
teacher_compatibility_verified = False
teacher_scoring_verified = False
gradient_flow_verified = False
optimizer_step_has_run = False
optimizer_step_verified = False
# The seed makes this sampled response reproducible on the same setup.
torch.manual_seed(42)

# We are only generating text. inference_mode() saves GPU memory because
# PyTorch does not need to track calculations for training.
with torch.inference_mode():
    complete_sequence_ids = student_model.generate(
        **prompt_inputs,
        # This limit counts response tokens only, not prompt tokens.
        max_new_tokens=128,
        # Sample from the student's own next-token choices. That makes the
        # response on-policy: it came from the student we are studying.
        do_sample=True,
        # Temperature 1.0 does not change those next-token probabilities.
        temperature=1.0,
        # Do not remove any candidate tokens before sampling.
        top_k=0,
        top_p=1.0,
        # Make one response to our one prompt.
        num_beams=1,
        num_return_sequences=1,
        pad_token_id=tokenizer.pad_token_id,
        # Return token IDs. We will calculate scores in Part 2.
        return_dict_in_generate=False,
        output_scores=False,
    )

# Shape: [1, prompt_length + response_length]. The prompt comes first;
# all later IDs belong to the student's newly generated response.

## 6. Separate and preserve the response tokens

`generate()` gives us one long row of IDs: **prompt IDs followed by response IDs**. We cut that row at `prompt_length`. The new tokens after the cut are the response we want to study. Their shape is `[1, response_length]`.

The checks make sure the prompt was kept and at least one response token was added. We turn the IDs back into readable text for display, but keep the original IDs for scoring. If the model generated an end token, its ID still counts even when the readable display hides it.

In [6]:
# generate() returns the original prompt followed by newly generated tokens.
# Therefore, everything beginning at prompt_length belongs to the response.
# The colon keeps the batch dimension: response_token_ids is [1, response_length].
response_token_ids = complete_sequence_ids[:, prompt_length:]
response_length = response_token_ids.shape[1]
total_sequence_length = complete_sequence_ids.shape[1]

# Check the boundary: prompt IDs are context; response IDs are what
# the student generated and what we will score later.
assert response_length > 0, "The student must generate at least one response token."
assert torch.equal(
    complete_sequence_ids[:, :prompt_length], prompt_token_ids
), "The complete sequence must begin with the original prompt token IDs."
assert response_length == total_sequence_length - prompt_length, (
    "The response-token count must equal total length minus prompt length."
)

# [0] selects our only response. Hide special tokens in the text display,
# but leave their original IDs unchanged for later scoring.
student_response_text = tokenizer.decode(
    response_token_ids[0], skip_special_tokens=True
)
print("Decoded student response:\n" + student_response_text)
print("Generated response token IDs:", response_token_ids[0].tolist())
print("Response-token tensor shape:", tuple(response_token_ids.shape))
print("Number of response tokens:", response_length)
print("Total sequence length (prompt + response):", total_sequence_length)

# Keep these IDs in memory. Decoding and tokenizing again might produce
# different IDs, so later parts reuse the originals.

Decoded student response:
 To determine the number of pencils remaining, follow these steps:

1. **Calculate the total number of pencils in the store:**
   
   The store has 24 boxes, and each box contains 15 pencils.
   
   \[
   \text{Total pencils} = 24 \text{ boxes} \times 15 \text{ pencils/box} = 360 \text{ pencils}
   \]

2. **Subtract the number of pencils sold:**
   
   The store sold 87 pencils.
   
   \[
   \text{Pencils remaining} = 3
Generated response token IDs: [2014, 8253, 279, 1372, 315, 96338, 9664, 11, 1795, 1493, 7354, 1447, 16, 13, 3070, 47866, 279, 2790, 1372, 315, 96338, 304, 279, 3553, 25, 1019, 5872, 256, 576, 3553, 702, 220, 17, 19, 14697, 11, 323, 1817, 3745, 5610, 220, 16, 20, 96338, 624, 5872, 256, 1124, 9640, 256, 1124, 1318, 90, 7595, 96338, 92, 284, 220, 17, 19, 1124, 1318, 90, 14697, 92, 1124, 15136, 220, 16, 20, 1124, 1318, 90, 96338, 14, 2011, 92, 284, 220, 18, 21, 15, 1124, 1318, 90, 96338, 532, 256, 1124, 2533, 17, 13, 3070, 3136, 2144, 279, 1372, 31

## What we accomplished

If the cells above run successfully:

- The student wrote one response to the question.
- `response_token_ids` holds the exact IDs it generated; the prompt IDs are kept separately.
- The student's weights have not changed. The teacher has not been used.
- In Part 2, we ask the student how likely it considered each token it wrote.

The prompt, the full sequence, and the response IDs stay in notebook memory for the next part.

# Part 2: Student token log probabilities

Run Part 1 first. Now we ask the **same student** how likely it thought each token in its response was. We reuse its exact token IDs; we do not turn the response back into text and tokenize it again.

`student` below is just a shorter name for `student_model`. This part reads the model's scores without changing its weights.


## 1. Pass the complete sequence through the student

`complete_sequence_ids` is the full row of tokens: prompt first, response second. Its shape is `[1, sequence_length]`, where `sequence_length = prompt_length + response_length`.

Passing this row through the student gives **logits**: raw scores for possible next tokens. The shape is `[1, sequence_length, vocabulary_size]`. At each position, there is one score for every token the student could choose next. The model can use earlier tokens but cannot look ahead at later ones.

`torch.no_grad()` tells PyTorch that we are only reading scores. We are not training the student.


In [ ]:
# A shorter name for the same model; no new model is loaded.
student = student_model
# New scores need to pass the checks below again.
student_scoring_verified = False

# Shape: [1, sequence_length]. All positions contain prompt or response
# tokens, so the attention mask is all ones. The model also hides future
# tokens from each position when it makes next-token predictions.
complete_attention_mask = torch.ones_like(complete_sequence_ids)

# Score the existing sequence without tracking training calculations.
# We do not need a generation cache for this one pass.
with torch.no_grad():
    student_forward_result = student(
        input_ids=complete_sequence_ids,
        attention_mask=complete_attention_mask,
        use_cache=False,
        return_dict=True,
    )

# logits[b, t, v] is the raw score for token v after position t in
# example b. These scores are not probabilities yet.
logits = student_forward_result.logits
batch_size, sequence_length, vocabulary_size = logits.shape
print("Logits shape [batch_size, sequence_length, vocabulary_size]:", tuple(logits.shape))
print("Dimension 0 - batch_size:", batch_size)
print("Dimension 1 - sequence_length (prompt + response):", sequence_length)
print("Dimension 2 - vocabulary_size:", vocabulary_size)


Logits shape [batch_size, sequence_length, vocabulary_size]: (1, 164, 151936)
Dimension 0 — batch_size: 1
Dimension 1 — sequence_length (prompt + response): 164
Dimension 2 — vocabulary_size: 151936


## 2. Align each prediction with its next-token target

A score at position `t` predicts the **next** token, at position `t + 1`. For example, the scores after the final prompt token predict the first response token. We line up scores and actual tokens by shifting them one place.

- `logits[:, :-1, :]` drops the final scores because there is no next token stored after them.
- `complete_sequence_ids[:, 1:]` drops the first token because there are no earlier scores to predict it.

Now `shifted_logits` has shape `[1, sequence_length - 1, vocabulary_size]`, and `target_token_ids` has shape `[1, sequence_length - 1]`. Position 0 in these two tensors pairs the scores after original token 0 with original token 1.


In [8]:
# Drop scores after the final stored token: no next token is available.
# Shape: [batch_size, sequence_length - 1, vocabulary_size].
shifted_logits = logits[:, :-1, :]

# Drop the first token and keep the tokens predicted by the scores above.
# Shape: [1, sequence_length - 1]. This still includes prompt tokens
# after the first one, plus every generated response token.
target_token_ids = complete_sequence_ids[:, 1:]

print("Shifted logits shape:", tuple(shifted_logits.shape))
print("Target token IDs shape:", tuple(target_token_ids.shape))


Shifted logits shape: (1, 163, 151936)
Target token IDs shape: (1, 163)


## 3. Select the log probability of each token that appeared

At each position, the student gives a score to **every possible next token**. `log_softmax` turns those raw scores into log probabilities. A larger log probability means the student considered that token more likely. We use `.float()` for this calculation because full precision makes it more stable; the model's weights stay in BF16 or FP16.

`all_token_log_probs` still has one value for every possible token. `target_token_ids` tells us which token actually appeared. `gather` picks that token's value at each position. See the PyTorch guides for [log-softmax](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html) and [gather](https://docs.pytorch.org/docs/stable/generated/torch.gather.html).

Each score uses the prompt and the earlier response tokens as context. This scores the response we already saved; it does not generate another one.


In [9]:
# Use full precision for the log-probability calculation. dim=-1 means
# compare possible next tokens, not positions in the response.
# Shape remains [batch_size, sequence_length - 1, vocabulary_size].
all_token_log_probs = torch.log_softmax(
    shifted_logits.float(),
    dim=-1,
)

# We need only the score of the token that actually appeared.
# unsqueeze(-1) changes target IDs to shape [1, sequence_length - 1, 1]
# so gather can select one vocabulary score per position. squeeze(-1)
# removes that last size-1 dimension, leaving [1, sequence_length - 1].
selected_token_log_probs = all_token_log_probs.gather(
    dim=-1,
    index=target_token_ids.unsqueeze(-1),
).squeeze(-1)

print("Selected token log-probability shape:", tuple(selected_token_log_probs.shape))


Selected token log-probability shape: (1, 163)


## 4. Keep only response-token scores

Now we keep scores for the **response only**. The first response token is at index `prompt_length` in the full sequence. The scores that predicted it came one step earlier, after the final prompt token at index **`prompt_length - 1`**.

Because we shifted the targets, its score is also at index `prompt_length - 1` in `selected_token_log_probs`. Starting the slice at `prompt_length` would lose that first response score.

| Original logits position | Original target token position | Token being scored |
| --- | --- | --- |
| `prompt_length - 1` | `prompt_length` | First response token |
| `prompt_length` | `prompt_length + 1` | Second response token, if present |

The slice removes scores **of prompt tokens** while keeping every score **of a response token**. Its shape should be `[1, response_length]`. We also turn log probabilities into ordinary probabilities for display.


In [10]:
# The first response token is at prompt_length in the full sequence.
# The final prompt position, prompt_length - 1, predicted it.
response_start_in_shifted_sequence = prompt_length - 1

# Drop prompt-token scores, but keep every response-token score.
# Shape: [1, response_length], including a generated end token if present.
response_student_log_probs = selected_token_log_probs[
    :,
    response_start_in_shifted_sequence:
]

# exp() turns log probabilities into ordinary probabilities for display.
# Shape is still [1, response_length]. This is not a training step.
response_student_probs = response_student_log_probs.exp()

print("Response log-probability shape:", tuple(response_student_log_probs.shape))
print("Expected response-token count:", response_length)


Response log-probability shape: (1, 128)
Expected response-token count: 128


## 5. Check token alignment and numerical values

These checks ask two things: **Do we have one score per response token?** And **does each score line up with the right token ID?** Checking the IDs catches a one-position mistake that a shape check alone might miss.

The scores also need to be valid numbers. Log probabilities should be at most zero, and ordinary probabilities should be between zero and one. The success message prints only if every check passes.


In [11]:
# Shifting removes one position. Each remaining token needs one score.
assert shifted_logits.shape[1] == complete_sequence_ids.shape[1] - 1
assert target_token_ids.shape == selected_token_log_probs.shape

# The response slice must have the right length and the exact generated
# token IDs, in order. This catches a one-position alignment mistake.
assert response_student_log_probs.shape[1] == response_length
assert torch.equal(
    target_token_ids[:, response_start_in_shifted_sequence:],
    response_token_ids,
)

# Every log probability must be a real, finite number at most zero.
# Every displayed ordinary probability must be between zero and one.
assert torch.isfinite(response_student_log_probs).all()
assert (response_student_log_probs <= 0).all()
assert (response_student_probs >= 0).all()
assert (response_student_probs <= 1).all()

# This flag exists only after every student scoring assertion above passes.
student_scoring_verified = True
print("All student token-alignment tests passed.")


All student token-alignment tests passed.


## 6. Inspect the response token by token

This table gives **one row to each response token**. Position 0 means the first token of the response, not the first token of the full prompt-plus-response sequence. We show each token as text; `repr()` makes spaces and newlines visible. The token ID is still the exact value used for scoring.

We display the first 30 rows but keep the full table in `response_token_table`. The mean, lowest, and highest scores use **all** response tokens. A high score means the student expected a token; it does not tell us whether the final answer is correct.


In [12]:
import pandas as pd
from IPython.display import display

# [0] selects our only response. Move its IDs and scores to CPU lists
# so pandas can display them. Each list has response_length items.
response_token_id_list = response_token_ids[0].cpu().tolist()
response_log_probability_list = response_student_log_probs[0].cpu().tolist()
response_probability_list = response_student_probs[0].cpu().tolist()

# Decode each ID on its own. repr() shows spaces and newline markers.
# Keep special tokens visible because they also have scores.
response_token_table = pd.DataFrame({
    "response_position": list(range(response_length)),
    "token_id": response_token_id_list,
    "decoded_token": [repr(tokenizer.decode([token_id])) for token_id in response_token_id_list],
    "student_log_probability": response_log_probability_list,
    "student_probability": response_probability_list,
})
display(response_token_table.head(30))

# These summaries use all response tokens, not only the first 30.
# item() turns a one-number tensor into a printable Python number.
print("Number of response tokens:", response_length)
print("Mean response-token log probability:", response_student_log_probs.mean().item())
print("Minimum response-token log probability:", response_student_log_probs.min().item())
print("Maximum response-token log probability:", response_student_log_probs.max().item())


,response_position,token_id,decoded_token,student_log_probability,student_probability
0,0,2014,' To',-6.868337e-01,0.503167
1,1,8253,' determine',-1.859482e-01,0.830317
2,2,279,' the',-2.694981e+00,0.067544
3,3,1372,' number',-2.930515e-03,0.997074
4,4,315,' of',0.000000e+00,1.000000
5,5,96338,' pencils',-1.159665e-02,0.988470
6,6,9664,' remaining',-2.841436e-01,0.752659
7,7,11,"','",-1.417507e+00,0.242317
8,8,1795,' follow',-1.328678e+00,0.264827
9,9,1493,' these',-3.308702e-04,0.999669


Number of response tokens: 128
Mean response-token log probability: -0.1179971992969513
Minimum response-token log probability: -2.6949806213378906
Maximum response-token log probability: 0.0


## 7. Manually inspect the prompt-to-response boundary

This display shows the most important alignment: **scores after the final prompt token → first response token**. Positions start at zero.

The student uses the whole prompt to make this prediction. We print the two tokens at the boundary, the position of the predicting scores, and the score given to the first response token.


In [13]:
# [0] is our only example. The prompt ends at prompt_length - 1;
# the response starts at prompt_length.
final_prompt_position = prompt_length - 1
first_response_position = prompt_length
final_prompt_token_id = complete_sequence_ids[0, final_prompt_position].item()
first_response_token_id = response_token_ids[0, 0].item()

# logits at the final prompt position predicted the first response ID.
# response_student_log_probs[0, 0] is that ID's chosen score.
first_response_student_log_probability = response_student_log_probs[0, 0].item()

print(
    f"Final prompt token: position {final_prompt_position}, ID {final_prompt_token_id}, "
    f"text {repr(tokenizer.decode([final_prompt_token_id]))}"
)
print(
    f"  logits at position {final_prompt_position}  -->  "
    f"first response token at position {first_response_position}"
)
print(
    f"First response token: ID {first_response_token_id}, "
    f"text {repr(tokenizer.decode([first_response_token_id]))}"
)
print("Student log probability of that first response token:", first_response_student_log_probability)


Final prompt token: position 35, ID 13, text '.'
  logits at position 35  -->  first response token at position 36
First response token: ID 2014, text ' To'
Student log probability of that first response token: -0.6868337392807007


## What this test proved

If Part 2 prints its success message:

- The student read the full prompt and response and predicted each next token.
- We kept the log probability it gave to each token it actually wrote.
- We removed scores for prompt tokens. `response_student_log_probs` has one score per response token.
- We checked the one-token shift and the numerical values. No weights changed.
- Part 3 checks whether the planned teacher can read those same token IDs.


# Part 3: Check that the teacher can read the student's tokens

Run Part 2 until you see **All student token-alignment tests passed.** We chose a larger model, `Qwen/Qwen3-32B`, to act as the teacher. The [GKD paper](https://arxiv.org/abs/2306.13649) asks the teacher to give feedback on a response generated by the student. Before loading a large teacher, we check that it reads the student's token IDs correctly.

A token ID is like a dictionary entry number. For direct token-by-token scoring, an ID used in this trajectory must mean the same piece of text to both models. We check the **whole student token mapping**, the prompt's tokenization, and every ID in the saved prompt and response. The teacher may have extra special tokens of its own; those do not change the meanings of the student IDs used here.

This cell downloads only small teacher files: its configuration, tokenizer, and weight index. It does **not** load the teacher's weights. The weight index lets us estimate whether the checkpoint can fit in GPU memory.

In [14]:
import json
from huggingface_hub import hf_hub_download

# Part 2 must have checked the student scores for this trajectory.
# Otherwise there is nothing reliable for the teacher to score next.
teacher_compatibility_verified = False
assert globals().get("student_scoring_verified", False), (
    "Run the Part 2 alignment checks before checking the teacher."
)
assert response_student_log_probs.shape == response_token_ids.shape
assert torch.isfinite(response_student_log_probs).all()

# Fetch model information only. The large teacher weights are not loaded here.
teacher_config_path = hf_hub_download(teacher_model_id, "config.json")
teacher_index_path = hf_hub_download(
    teacher_model_id, "model.safetensors.index.json"
)
teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_model_id)
with open(teacher_config_path, encoding="utf-8") as config_file:
    teacher_config = json.load(config_file)
with open(teacher_index_path, encoding="utf-8") as index_file:
    teacher_weight_index = json.load(index_file)

# The model's output has one score for each vocabulary position.
# Both output heads must have the same number of positions for later work.
student_vocabulary_size = student.config.vocab_size
teacher_vocabulary_size = teacher_config["vocab_size"]
assert student_vocabulary_size == logits.shape[-1]
assert teacher_vocabulary_size == student_vocabulary_size

# The checkpoint size is an estimate of the weight memory, not a promise
# that the model will fit: temporary calculations also need GPU memory.
teacher_checkpoint_gib = (
    teacher_weight_index["metadata"]["total_size"] / 1024**3
)
print("Student output vocabulary size:", student_vocabulary_size)
print("Teacher output vocabulary size:", teacher_vocabulary_size)
print("Teacher architecture:", teacher_config["architectures"][0])
print(f"Teacher checkpoint size: {teacher_checkpoint_gib:.1f} GiB")
print(f"GPU memory (total): {gpu_memory_gib:.1f} GiB")

# get_vocab() maps each text piece to its integer token ID.
# Every student entry must have the SAME number in the teacher's map.
# Extra teacher-only special tokens are allowed for this scoring step.
student_token_vocabulary = tokenizer.get_vocab()
teacher_token_vocabulary = teacher_tokenizer.get_vocab()
student_mapping_mismatches = [
    (piece, token_id, teacher_token_vocabulary.get(piece))
    for piece, token_id in student_token_vocabulary.items()
    if teacher_token_vocabulary.get(piece) != token_id
]
teacher_only_token_count = len(
    teacher_token_vocabulary.keys() - student_token_vocabulary.keys()
)
print("Student token mapping mismatches:", len(student_mapping_mismatches))
print("Teacher-only tokenizer entries:", teacher_only_token_count)
assert not student_mapping_mismatches, (
    "The teacher assigns different IDs to some student tokens."
)

# The teacher must also split THIS prompt into the same IDs.
# Shape is [1, prompt_length] because we have one example.
teacher_prompt_token_ids = teacher_tokenizer(
    prompt_text, add_special_tokens=False
)["input_ids"]
assert teacher_prompt_token_ids == prompt_token_ids[0].tolist()

# Check every ID in the exact student-generated sequence. The teacher
# must read the same prompt and response pieces at these positions.
# complete_sequence_ids has shape [1, prompt_length + response_length].
token_piece_mismatches = []
for sequence_position, token_id in enumerate(complete_sequence_ids[0].tolist()):
    student_piece = tokenizer.convert_ids_to_tokens(token_id)
    teacher_piece = teacher_tokenizer.convert_ids_to_tokens(token_id)
    if student_piece != teacher_piece:
        token_piece_mismatches.append(
            (sequence_position, token_id, student_piece, teacher_piece)
        )

prompt_mismatch_count = sum(
    position < prompt_length for position, *_ in token_piece_mismatches
)
response_mismatch_count = len(token_piece_mismatches) - prompt_mismatch_count
print("Prompt ID meaning mismatches:", prompt_mismatch_count)
print("Response ID meaning mismatches:", response_mismatch_count)
assert not token_piece_mismatches, (
    "The teacher would read different text from these student token IDs."
)

# Part 4 may load the teacher only after all compatibility checks pass.
teacher_compatibility_verified = True
print("Teacher tokenizer compatibility checks passed.")


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Student output vocabulary size: 151936
Teacher output vocabulary size: 151936
Teacher architecture: Qwen3ForCausalLM
Teacher checkpoint size: 61.0 GiB
GPU memory (total): 79.3 GiB
Student token mapping mismatches: 0
Teacher-only tokenizer entries: 0
Prompt ID meaning mismatches: 0
Response ID meaning mismatches: 0
Teacher tokenizer compatibility checks passed.


## What this checkpoint tells us

If you see **Teacher tokenizer compatibility checks passed**, the teacher gives the same meanings to the student's token IDs in this response. We have also checked the student token mapping and the prompt's tokenization. The teacher may know some extra special tokens; that matters later if we compare complete next-token distributions, but it does not stop us from checking the scores of the response IDs we already have.

We have still downloaded **no teacher weights** and changed **no model weights**. The next part loads the frozen teacher and asks it to score those same response tokens.

# Part 4: Let the frozen teacher score the student's response

The student has already written the response. The teacher does **not** write a new one. We give it the exact `complete_sequence_ids`: prompt IDs first, then the student's response IDs. At each position, the teacher sees the prompt and earlier response tokens and predicts what could come next.

The [GKD paper](https://arxiv.org/abs/2306.13649) uses teacher feedback on student-generated sequences. This part only checks that we can obtain one teacher score for each token the student actually wrote. It does **not** calculate the paper's divergence or train the student.

`Qwen/Qwen3-32B` is large. Its BF16 checkpoint is about 61 GiB, so this part is intended for an A100 80GB runtime with enough free GPU memory and disk space. It will stop before downloading the weights if the free GPU memory is clearly too small.

## 1. Load and freeze the teacher

**Frozen** means the teacher's weights will not change. `eval()` turns off training-only behavior, and `requires_grad_(False)` prevents PyTorch from preparing teacher-weight gradients. We check available GPU memory before the large download. The 8 GiB margin allows room beyond the checkpoint for the student and temporary work; it is a safety estimate, not a guarantee that Colab will have enough memory.

In [15]:
# A new teacher run must pass all checks below before it is considered verified.
teacher_scoring_verified = False
assert globals().get("teacher_compatibility_verified", False), (
    "Run Part 3 and pass the tokenizer checks before loading the teacher."
)

# Clear unused cached GPU blocks before checking free memory.
# This does not delete the student's model or saved trajectory.
torch.cuda.empty_cache()
free_gpu_bytes, _ = torch.cuda.mem_get_info(device)
free_gpu_gib = free_gpu_bytes / 1024**3
minimum_headroom_gib = 8.0
print(f"Free GPU memory before teacher load: {free_gpu_gib:.1f} GiB")
print(f"Estimated teacher weights: {teacher_checkpoint_gib:.1f} GiB")
assert free_gpu_gib >= teacher_checkpoint_gib + minimum_headroom_gib, (
    "Not enough free GPU memory for this BF16 teacher plus a safety margin. "
    "Use an A100 80GB runtime or a smaller teacher."
)

# Load all teacher weights onto the same GPU as the saved token IDs.
# low_cpu_mem_usage reduces temporary CPU copies while loading shards.
teacher = AutoModelForCausalLM.from_pretrained(
    teacher_model_id,
    torch_dtype=model_dtype,
    device_map={"": device.index},
    low_cpu_mem_usage=True,
)
teacher.eval()
teacher.requires_grad_(False)
assert not teacher.training
assert not any(parameter.requires_grad for parameter in teacher.parameters())
print("Frozen teacher loaded:", teacher_model_id)
print("Teacher weight dtype:", next(teacher.parameters()).dtype)


Free GPU memory before teacher load: 77.5 GiB
Estimated teacher weights: 61.0 GiB


Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

model-00005-of-00017.safetensors:   0%|          | 0.00/3.90G [00:00<?, ?B/s]

model-00008-of-00017.safetensors:   0%|          | 0.00/3.90G [00:00<?, ?B/s]

model-00001-of-00017.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

...

model-00016-of-00017.safetensors:   0%|          | 0.00/3.90G [00:00<?, ?B/s]

model-00017-of-00017.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/17 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Frozen teacher loaded: Qwen/Qwen3-32B
Teacher weight dtype: torch.bfloat16


## 2. Ask the teacher to read the saved sequence

We pass the **same** `complete_sequence_ids` used for student scoring through the teacher once. Its shape is `[1, sequence_length]`, where `sequence_length = prompt_length + response_length`. We do not retokenize the response or sample from the teacher.

The teacher returns raw scores called **logits**, with shape `[1, sequence_length, teacher_vocabulary_size]`. The score at position `t` predicts the token at position `t + 1`. `torch.no_grad()` makes this a read-only scoring pass.

In [16]:
# Check that these are still the original student-generated IDs.
assert torch.equal(complete_sequence_ids[:, :prompt_length], prompt_token_ids)
assert torch.equal(complete_sequence_ids[:, prompt_length:], response_token_ids)

# All positions are real tokens, so the attention mask is all ones.
# Shape: [1, sequence_length]. The model still hides future positions.
teacher_attention_mask = torch.ones_like(complete_sequence_ids)
with torch.no_grad():
    teacher_forward_result = teacher(
        input_ids=complete_sequence_ids,
        attention_mask=teacher_attention_mask,
        use_cache=False,
        return_dict=True,
    )

# Shape: [1, sequence_length, teacher_vocabulary_size].
# There is a raw score for each possible next token at each position.
teacher_logits = teacher_forward_result.logits
print(
    "Teacher logits shape [batch, sequence, vocabulary]:",
    tuple(teacher_logits.shape),
)
print("Batch size:", teacher_logits.shape[0])
print("Prompt + response positions:", teacher_logits.shape[1])
print("Teacher vocabulary positions:", teacher_logits.shape[2])


Teacher logits shape [batch, sequence, vocabulary]: (1, 164, 151936)
Batch size: 1
Prompt + response positions: 164
Teacher vocabulary positions: 151936


## 3. Line up predictions with response tokens

We shift by one position just as we did for the student. The teacher's logits at the **last prompt position**, `prompt_length - 1`, predicted the **first response token**, at `prompt_length`. We remove the final logits position because we have no following token to check, and remove the first token ID because no earlier logits predicted it.

Then we start at `prompt_length - 1` in the shifted scores. This excludes prompt-token scores while keeping the first and every later response-token score. The teacher is judging the exact student trajectory, not a teacher-generated answer.

In [17]:
# Shape: [1, sequence_length - 1, teacher_vocabulary_size].
# Position t predicts the token that appears at position t + 1.
teacher_shifted_logits = teacher_logits[:, :-1, :]
teacher_target_token_ids = complete_sequence_ids[:, 1:]

# The first response token is at complete-sequence index prompt_length.
# It was predicted one step earlier, at prompt_length - 1.
teacher_response_start = prompt_length - 1
teacher_response_logits = teacher_shifted_logits[
    :, teacher_response_start:, :
]
teacher_response_target_ids = teacher_target_token_ids[
    :, teacher_response_start:
]

# Convert BF16/FP16 raw scores to stable FP32 log probabilities.
# At each response position, gather only the score of the token the
# student actually generated. Shape becomes [1, response_length].
teacher_response_all_log_probs = torch.log_softmax(
    teacher_response_logits.float(),
    dim=-1,
)
teacher_response_log_probs = teacher_response_all_log_probs.gather(
    dim=-1,
    index=teacher_response_target_ids.unsqueeze(-1),
).squeeze(-1)

print("Teacher response logits shape:", tuple(teacher_response_logits.shape))
print("Teacher response target IDs shape:", tuple(teacher_response_target_ids.shape))
print("Teacher response log-probability shape:", tuple(teacher_response_log_probs.shape))


Teacher response logits shape: (1, 128, 151936)
Teacher response target IDs shape: (1, 128)
Teacher response log-probability shape: (1, 128)


## 4. Check the scores and inspect a few tokens

We should have **one teacher log probability for every student-generated response token**, in the same order. The checks below confirm the sequence length, the one-token shift, exact token IDs, finite numbers, and frozen teacher weights. We display a few teacher scores by themselves; comparing student and teacher scores is a later part. A high score means the teacher expected a token, not that the whole answer is correct.

In [18]:
# The one-token shift removes exactly one sequence position.
assert teacher_logits.shape[:2] == complete_sequence_ids.shape
assert teacher_logits.shape[-1] == teacher_vocabulary_size
assert teacher_shifted_logits.shape[1] == complete_sequence_ids.shape[1] - 1
assert teacher_target_token_ids.shape == complete_sequence_ids[:, 1:].shape

# The slice begins at the logits after the last prompt token.
# These target IDs must be exactly the student's response IDs.
assert teacher_response_start == prompt_length - 1
assert teacher_response_target_ids.shape == response_token_ids.shape
assert torch.equal(teacher_response_target_ids, response_token_ids)
assert teacher_response_log_probs.shape == response_token_ids.shape
assert teacher_response_log_probs.shape[1] == response_length

# Log probabilities must be finite and at most zero.
# No teacher gradients or weight updates were made.
assert torch.isfinite(teacher_response_log_probs).all()
assert (teacher_response_log_probs <= 0).all()
assert not teacher_response_log_probs.requires_grad
assert not any(parameter.requires_grad for parameter in teacher.parameters())

# [0] selects our only example. repr() makes spaces and newlines visible.
teacher_token_table = pd.DataFrame({
    "response_position": list(range(response_length)),
    "token_id": response_token_ids[0].cpu().tolist(),
    "decoded_token": [
        repr(tokenizer.decode([token_id]))
        for token_id in response_token_ids[0].cpu().tolist()
    ],
    "teacher_log_probability": teacher_response_log_probs[0].cpu().tolist(),
})
display(teacher_token_table.head(20))
print("Number of teacher-scored response tokens:", response_length)
print(
    "Teacher log probability of the first response token:",
    teacher_response_log_probs[0, 0].item(),
)
print(
    "Logits position that predicted the first response token:",
    teacher_response_start,
)

# Print success only after all checks above pass.
teacher_scoring_verified = True
print("All frozen-teacher token-scoring tests passed.")


,response_position,token_id,decoded_token,teacher_log_probability
0,0,2014,' To',-0.924340
1,1,8253,' determine',-0.849176
2,2,279,' the',-1.702581
3,3,1372,' number',-0.350088
4,4,315,' of',-0.000096
5,5,96338,' pencils',-0.129048
6,6,9664,' remaining',-0.136605
7,7,11,"','",-1.427624
8,8,1795,' follow',-2.539629
9,9,1493,' these',-0.016024


Number of teacher-scored response tokens: 128
Teacher log probability of the first response token: -0.9243395328521729
Logits position that predicted the first response token: 35
All frozen-teacher token-scoring tests passed.


## What this part proved

If you see **All frozen-teacher token-scoring tests passed**:

- The student still generated the response. The teacher did not generate a new one.
- The frozen teacher read the same prompt and response token IDs.
- Its prediction after the final prompt token scored the first response token.
- We kept one finite teacher log probability for every student response token.
- Prompt-token scores were left out, and no model weights changed.

This is one short demonstration response. The 128-token limit may stop it before the answer is complete. We have now checked student and teacher scores **separately**. We have not compared them, calculated a GKD divergence, or trained the student. Those are later steps. In the saved A100 run, both tokenizer mappings matched. We can now compare the models' full next-token distributions at the response positions.

# Part 5: Compare student and teacher with reverse KL

The [GKD paper](https://arxiv.org/html/2306.13649v3#S3) compares the teacher and student **next-token distributions** at each position of a student-generated response. A distribution contains a probability for every possible next token. In this demonstration we choose **reverse KL**, one of the divergences studied in the paper:

`KL(student || teacher) = sum_over_vocabulary student_probability × (student_log_probability − teacher_log_probability)`

A single generated token has one student score and one teacher score, but their difference is **not** the KL divergence. KL compares all possible next tokens at that same position. We will average the response-position KL values to get one number for this one sampled trajectory. The prompt provides context; its token positions are not included in the average. This is a measurement only: no gradients or training.


## 1. Reuse the same response positions

Parts 2 and 4 already ran both models on the same complete prompt-plus-response sequence. We reuse those logits; no new response is generated. The first response token is at full-sequence position `prompt_length`, but both models predicted it from position `prompt_length - 1`. Slicing the student and teacher logits at this boundary gives one next-token distribution for each response token. Each tensor has shape `[1, response_length, vocabulary_size]`.


In [19]:
# These flags were set only after the earlier token and score checks passed.
# A fresh runtime must run Parts 1–4 before this comparison is meaningful.
assert globals().get("student_scoring_verified", False)
assert globals().get("teacher_compatibility_verified", False)
assert globals().get("teacher_scoring_verified", False)

# shifted_logits[t] predicts target_token_ids[t], which is the token at
# complete_sequence_ids position t + 1. The first response target therefore
# uses logits at prompt_length - 1, immediately after the final prompt token.
student_response_logits = shifted_logits[
    :, response_start_in_shifted_sequence:, :
]

# Both models saw the same prefixes, and the tokenizer checks established
# that their shared vocabulary IDs mean the same thing. The batch size is 1.
expected_response_logits_shape = (
    response_token_ids.shape[0], response_length, student_vocabulary_size
)
assert student_response_logits.shape == expected_response_logits_shape
assert teacher_response_logits.shape == expected_response_logits_shape
assert torch.equal(teacher_response_target_ids, response_token_ids)
assert response_start_in_shifted_sequence == teacher_response_start == prompt_length - 1

print("Student response logits [batch, response positions, vocabulary]:", tuple(student_response_logits.shape))
print("Teacher response logits [batch, response positions, vocabulary]:", tuple(teacher_response_logits.shape))
print("First response token is predicted from full-sequence logits position:", prompt_length - 1)


Student response logits [batch, response positions, vocabulary]: (1, 128, 151936)
Teacher response logits [batch, response positions, vocabulary]: (1, 128, 151936)
First response token is predicted from full-sequence logits position: 35


## 2. Calculate reverse KL at every response position

`log_softmax` turns raw logits into log probabilities over the full vocabulary. We use FP32 for this calculation even though the models were loaded in BF16. We then take `exp()` to get the student's ordinary probabilities. For each response position, reverse KL weights the log-probability difference by the **student's** probability for every vocabulary token, then sums those weighted differences. The result has shape `[1, response_length]`.

We use `torch.no_grad()` because this checkpoint only measures the two existing models. A future training step will need a new student forward pass with gradients enabled. The teacher will stay frozen.


In [20]:
# Reuse the frozen teacher's full-vocabulary log probabilities from Part 4.
# Shape: [1, response_length, vocabulary_size]. This is different from
# teacher_response_log_probs, which contains only the generated token's score.
assert teacher_response_all_log_probs.shape == teacher_response_logits.shape

with torch.no_grad():
    # FP32 improves numerical stability before normalizing over vocabulary.
    student_response_all_log_probs = torch.log_softmax(
        student_response_logits.float(), dim=-1
    )
    student_response_all_probs = student_response_all_log_probs.exp()

    # At each response position, compare ALL possible next tokens.
    # Sum over vocabulary (dim=-1), leaving shape [1, response_length].
    response_reverse_kl = (
        student_response_all_probs
        * (student_response_all_log_probs - teacher_response_all_log_probs)
    ).sum(dim=-1)

    # The paper averages token-level divergences over response length.
    # This scalar describes only our one sampled trajectory.
    mean_response_reverse_kl = response_reverse_kl.mean()

print("Reverse KL tensor shape [batch, response positions]:", tuple(response_reverse_kl.shape))
print("Mean reverse KL for this response (nats):", mean_response_reverse_kl.item())


Reverse KL tensor shape [batch, response positions]: (1, 128)
Mean reverse KL for this response (nats): 0.1592046320438385


## 3. Check the calculation and inspect the result

The checks below confirm that the full distributions match the token scores we already verified, each student distribution sums to about 1, and the KL values are finite. Exact KL cannot be negative; tiny negative values can arise from floating-point rounding, so we allow a small numerical tolerance. The table shows the score each model gave the token that actually appeared and the **full-vocabulary** reverse KL at that response position.


In [21]:
# Each full distribution must reproduce the selected-token scores from
# Parts 2 and 4. This catches a response-boundary or one-token-shift mistake.
student_selected_again = student_response_all_log_probs.gather(
    dim=-1, index=response_token_ids.unsqueeze(-1)
).squeeze(-1)
teacher_selected_again = teacher_response_all_log_probs.gather(
    dim=-1, index=response_token_ids.unsqueeze(-1)
).squeeze(-1)
assert torch.allclose(student_selected_again, response_student_log_probs, atol=1e-4)
assert torch.allclose(teacher_selected_again, teacher_response_log_probs, atol=1e-4)

# One probability distribution and one KL value belong to each generated
# response token. The prompt positions have already been excluded.
assert response_reverse_kl.shape == response_token_ids.shape
assert torch.isfinite(student_response_all_log_probs).all()
assert torch.isfinite(teacher_response_all_log_probs).all()
assert torch.isfinite(response_reverse_kl).all()
assert torch.allclose(
    student_response_all_probs.sum(dim=-1),
    torch.ones_like(response_reverse_kl),
    atol=1e-4,
)
assert (response_reverse_kl >= -1e-4).all()
assert not response_reverse_kl.requires_grad
assert not any(parameter.requires_grad for parameter in teacher.parameters())

# [0] selects our one trajectory. One row corresponds to one response token;
# repr() makes a leading space or newline in decoded text visible.
reverse_kl_table = pd.DataFrame({
    "response_position": list(range(response_length)),
    "token_id": response_token_ids[0].cpu().tolist(),
    "decoded_token": [
        repr(tokenizer.decode([token_id]))
        for token_id in response_token_ids[0].cpu().tolist()
    ],
    "student_generated_token_log_prob": response_student_log_probs[0].cpu().tolist(),
    "teacher_generated_token_log_prob": teacher_response_log_probs[0].cpu().tolist(),
    "reverse_kl_nats": response_reverse_kl[0].cpu().tolist(),
})
display(reverse_kl_table.head(20))
print("Number of compared response positions:", response_length)
print("Mean reverse KL (nats):", mean_response_reverse_kl.item())
print("Minimum reverse KL (nats):", response_reverse_kl.min().item())
print("Maximum reverse KL (nats):", response_reverse_kl.max().item())
print("All reverse-KL alignment and numerical checks passed.")


,response_position,token_id,decoded_token,student_generated_token_log_prob,teacher_generated_token_log_prob,reverse_kl_nats
0,0,2014,' To',-6.868337e-01,-0.924340,1.228260
1,1,8253,' determine',-1.859482e-01,-0.849176,0.361974
2,2,279,' the',-2.694981e+00,-1.702581,0.056564
3,3,1372,' number',-2.930515e-03,-0.350088,0.333285
4,4,315,' of',0.000000e+00,-0.000096,0.000095
5,5,96338,' pencils',-1.159665e-02,-0.129048,0.090280
6,6,9664,' remaining',-2.841436e-01,-0.136605,0.063808
7,7,11,"','",-1.417507e+00,-1.427624,0.177608
8,8,1795,' follow',-1.328678e+00,-2.539629,0.209205
9,9,1493,' these',-3.308702e-04,-0.016024,0.014839


Number of compared response positions: 128
Mean reverse KL (nats): 0.1592046320438385
Minimum reverse KL (nats): -1.5509192152762807e-08
Maximum reverse KL (nats): 6.793417453765869
All reverse-KL alignment and numerical checks passed.


## What this part proved

If you see **All reverse-KL alignment and numerical checks passed**:

- The student generated the trajectory; the teacher scored that same trajectory.
- We compared both models' **full next-token distributions** only at response positions. The final prompt token predicts the first response token.
- Reverse KL is one number per response position, averaged over this one trajectory. It is not the difference between the two scores for the generated token alone.
- No weights were updated. One sampled response and one KL average are a demonstration, not evidence that the student has improved.

The paper's on-policy idea is to repeat this process as the student changes: sample from the current student, ask the teacher about those exact prefixes, and update the student using a chosen divergence. **We have not implemented that update yet.** The next checkpoint can add a student training forward pass and verify gradients before any optimizer step.


# Part 6: Check that reverse KL can train the student

Part 5 measured reverse KL inside `torch.no_grad()`. That was useful for checking the mathematics, but a value calculated without an autograd graph cannot teach the student. The next small milestone is to rebuild the same student calculation **with gradient tracking enabled** and call `backward()`.

This mirrors an important detail in the [GKD paper](https://arxiv.org/html/2306.13649v3#S3.SS1): the response is sampled from the current student, but gradients do not pass backward through the sampling operation. We treat the saved token IDs as fixed data. The frozen teacher supplies a fixed target distribution, and gradients pass only through a fresh student forward pass.

We still do **not** create an optimizer or update any weights in this part. We only prove that a usable learning signal reaches the student.


## 1. Prepare fixed trajectory and teacher targets

The tensor `complete_sequence_ids` contains the prompt followed by the student's sampled response. Token IDs are integers, so they are fixed choices rather than differentiable values. This is how we avoid backpropagating through sampling. Because Part 1 created this tensor inside `torch.inference_mode()`, PyTorch also marked it as an **inference tensor**. An embedding layer must save its input IDs for backward, so we clone the IDs outside inference mode to make an ordinary, autograd-safe tensor containing the exact same values.

The teacher probabilities are also fixed. `detach()` makes that boundary explicit: the teacher provides labels, but it receives no gradients. We clear any old student gradients so this test begins from a known state. The student stays in evaluation mode to make this check deterministic; evaluation mode and gradient tracking are separate ideas in PyTorch.


In [24]:
# Earlier parts must pass before we trust this trajectory, its alignment,
# or the teacher distributions used as fixed training targets.
gradient_flow_verified = False
assert globals().get("student_scoring_verified", False)
assert globals().get("teacher_scoring_verified", False)
assert response_reverse_kl.shape == response_token_ids.shape
assert torch.equal(
    complete_sequence_ids[:, prompt_length:], response_token_ids
)

# Part 1 created complete_sequence_ids inside inference_mode(). Such a
# tensor is fine for inference but cannot be saved by an embedding layer
# for backward. clone() here creates ordinary tensors with identical values.
training_complete_sequence_ids = complete_sequence_ids.detach().clone()
training_complete_attention_mask = complete_attention_mask.detach().clone()
assert torch.equal(training_complete_sequence_ids, complete_sequence_ids)
assert torch.equal(training_complete_attention_mask, complete_attention_mask)
assert not torch.is_inference(training_complete_sequence_ids)
assert not torch.is_inference(training_complete_attention_mask)

# eval() gives us the same deterministic model behavior used in Part 5.
# It does not disable autograd; only no_grad() or inference_mode() do that.
student.eval()
student.zero_grad(set_to_none=True)
assert all(parameter.grad is None for parameter in student.parameters())

# Shape: [1, response_length, vocabulary_size]. The teacher tensor is
# a fixed label for every possible next token at every response position.
teacher_training_target_log_probs = teacher_response_all_log_probs.detach()
assert not teacher_training_target_log_probs.requires_grad
assert not any(parameter.requires_grad for parameter in teacher.parameters())
assert all(parameter.grad is None for parameter in teacher.parameters())

# backward() should not change weights by itself. Save one small student
# parameter so we can directly confirm that its value stays unchanged.
student_reference_name, student_reference_parameter = min(
    (
        (name, parameter)
        for name, parameter in student.named_parameters()
        if parameter.requires_grad
    ),
    key=lambda named_parameter: named_parameter[1].numel(),
)
student_reference_before_backward = (
    student_reference_parameter.detach().clone()
)

print("Saved trajectory shape [batch, prompt + response]:", tuple(training_complete_sequence_ids.shape))
print("Autograd-safe token IDs created: True")
print("Fixed teacher target shape:", tuple(teacher_training_target_log_probs.shape))
print("Small student parameter used for the no-update check:", student_reference_name)


Saved trajectory shape [batch, prompt + response]: (1, 164)
Autograd-safe token IDs created: True
Fixed teacher target shape: (1, 128, 151936)
Small student parameter used for the no-update check: model.layers.0.self_attn.q_norm.weight


## 2. Recompute reverse KL with student gradients enabled

We pass the autograd-safe clone of the same complete sequence through the student again, this time without `torch.no_grad()`. The clone has identical integer token IDs; it is not a newly generated response. The student builds an autograd graph that records how its weights produced its logits.

We apply the same one-token shift and the same response-only slice used earlier. The loss is the mean of `KL(student || teacher)` across the 128 response positions. It should be numerically close to Part 5's value, but now `training_reverse_kl_loss.requires_grad` must be `True`.


In [25]:
# Do not wrap this forward pass in no_grad(): this is the student path
# whose operations PyTorch must remember for backward().
training_student_result = student(
    input_ids=training_complete_sequence_ids,
    attention_mask=training_complete_attention_mask,
    use_cache=False,
    return_dict=True,
)
training_student_logits = training_student_result.logits

# Position t predicts the token at t + 1. After removing the final logits
# position, prompt_length - 1 is still where the first response is predicted.
training_shifted_logits = training_student_logits[:, :-1, :]
training_response_logits = training_shifted_logits[
    :, response_start_in_shifted_sequence:, :
]
assert training_response_logits.shape == teacher_training_target_log_probs.shape
assert training_response_logits.shape[:2] == response_token_ids.shape

# FP32 log_softmax is more numerically stable than doing this reduction in
# BF16. Autograd still connects these values back to the BF16 student weights.
training_student_log_probs = torch.log_softmax(
    training_response_logits.float(), dim=-1
)
training_student_probs = training_student_log_probs.exp()

# Reverse KL weights each full-vocabulary log-probability difference by
# the student's probability. Shape after summing vocabulary: [1, 128].
training_response_reverse_kl = (
    training_student_probs
    * (training_student_log_probs - teacher_training_target_log_probs)
).sum(dim=-1)
training_reverse_kl_loss = training_response_reverse_kl.mean()

assert training_response_reverse_kl.shape == response_token_ids.shape
assert training_reverse_kl_loss.requires_grad
assert training_reverse_kl_loss.grad_fn is not None
assert torch.isfinite(training_reverse_kl_loss)
# With eval-mode behavior and unchanged weights, this fresh calculation
# should match the read-only result from Part 5 up to small rounding error.
assert torch.allclose(
    training_reverse_kl_loss.detach(),
    mean_response_reverse_kl,
    atol=1e-4,
    rtol=1e-4,
)

print("Training response KL shape:", tuple(training_response_reverse_kl.shape))
print("Differentiable mean reverse KL (nats):", training_reverse_kl_loss.item())
print("Loss tracks gradients:", training_reverse_kl_loss.requires_grad)


Training response KL shape: (1, 128)
Differentiable mean reverse KL (nats): 0.1592046320438385
Loss tracks gradients: True


## 3. Backpropagate and inspect the student gradients

Calling `backward()` applies the chain rule and fills each participating student parameter's `.grad` field. A gradient tells us the direction in which a small weight change would change the loss. It does **not** change the weight itself; that requires a later `optimizer.step()`.

We verify that at least one student parameter received a finite, nonzero gradient, that the teacher received none, and that the saved student parameter did not change. We then clear the student gradients to release memory.


In [26]:
# backward() calculates gradients for the student graph created above.
# There is deliberately no optimizer and no optimizer.step() in this part.
training_reverse_kl_loss.backward()

student_gradient_norm_squares = []
student_parameters_with_gradients = 0
student_parameters_with_nonzero_gradients = 0
for parameter in student.parameters():
    if parameter.grad is None:
        continue

    # Each gradient tensor must contain ordinary finite numbers. Calculate
    # its norm in FP32, then combine the small per-parameter norm values.
    assert torch.isfinite(parameter.grad).all()
    parameter_gradient_norm = parameter.grad.detach().float().norm()
    student_gradient_norm_squares.append(parameter_gradient_norm.square())
    student_parameters_with_gradients += 1
    if parameter_gradient_norm.item() > 0:
        student_parameters_with_nonzero_gradients += 1

assert student_parameters_with_gradients > 0
assert student_parameters_with_nonzero_gradients > 0
student_gradient_l2_norm = torch.stack(
    student_gradient_norm_squares
).sum().sqrt()
assert torch.isfinite(student_gradient_l2_norm)
assert student_gradient_l2_norm.item() > 0

# The teacher is frozen, and backward() alone must leave every model weight
# unchanged. The small saved student parameter makes that second fact visible.
assert all(parameter.grad is None for parameter in teacher.parameters())
assert torch.equal(
    student_reference_parameter.detach(),
    student_reference_before_backward,
)

gradient_flow_verified = True
print("Student parameter tensors with gradients:", student_parameters_with_gradients)
print("Student parameter tensors with nonzero gradients:", student_parameters_with_nonzero_gradients)
print("Student gradient L2 norm:", student_gradient_l2_norm.item())
print("Teacher parameter tensors with gradients: 0")
print("Reference student weight unchanged: True")
print("All student-only gradient-flow tests passed.")

# Gradients have served their purpose in this test. Clearing them releases
# memory and ensures the next part does not accidentally reuse them.
student.zero_grad(set_to_none=True)
assert all(parameter.grad is None for parameter in student.parameters())


Student parameter tensors with gradients: 310
Student parameter tensors with nonzero gradients: 310
Student gradient L2 norm: 18.54865074157715
Teacher parameter tensors with gradients: 0
Reference student weight unchanged: True
All student-only gradient-flow tests passed.


## What this part proved

If you see **All student-only gradient-flow tests passed**:

- The sampled response token IDs were treated as fixed on-policy data. We did not try to differentiate through token sampling.
- The frozen teacher's full next-token distributions were treated as fixed targets.
- A fresh student forward pass produced a differentiable mean reverse-KL loss.
- `backward()` produced finite, nonzero student gradients and no teacher gradients.
- No optimizer step was performed, so no student or teacher weight was updated.

In simple terms, we have now built a valid learning signal and confirmed that it points only toward the student. The next checkpoint can perform **one carefully controlled optimizer step**, compare the loss and a student weight before and after, and then explain why a complete experiment must repeatedly generate fresh trajectories from the newly updated student.


# Part 7: Perform one controlled student update

We have generated one student trajectory, obtained frozen-teacher targets, calculated reverse KL, and verified student-only gradients. We can now perform the final operation in one miniature GKD iteration: update the student once.

We use AdamW with a small demonstration learning rate and no weight decay. This step uses the same trajectory that the student sampled **before** the update, which is correct for this iteration. Once the student changes, that trajectory is no longer fully on-policy for another update. A real training loop must sample fresh responses from the updated student.

This is still an educational one-example test. We do not save a checkpoint, claim improved reasoning, or repeat the update. Restarting the runtime restores the original model from Hugging Face.


## 1. Create a guarded optimizer

An optimizer converts gradients into weight changes. We include only student parameters; the teacher cannot be updated because it is frozen and is not given to the optimizer. `foreach=False` uses less temporary GPU memory, which matters while the 32B teacher is still loaded.

The guard variable prevents a second update if this section is accidentally rerun. To intentionally repeat the complete experiment, restart the runtime and run the notebook from the top so the current student generates a new trajectory.


In [ ]:
# Part 6 must prove that this exact loss sends gradients only to the student.
assert globals().get("gradient_flow_verified", False)
assert not globals().get("optimizer_step_has_run", False), (
    "The one-step update already ran. Restart the runtime and run from the "
    "top before intentionally performing another on-policy iteration."
)
optimizer_step_verified = False
assert all(parameter.grad is None for parameter in student.parameters())
assert all(parameter.grad is None for parameter in teacher.parameters())

# This learning rate is for a single, observable demonstration update.
# A real experiment must tune it using held-out evaluation data.
demonstration_learning_rate = 1e-4
student_optimizer = torch.optim.AdamW(
    (parameter for parameter in student.parameters() if parameter.requires_grad),
    lr=demonstration_learning_rate,
    weight_decay=0.0,
    foreach=False,
)

# Save a small teacher parameter so we can directly prove it did not move.
teacher_reference_name, teacher_reference_parameter = min(
    teacher.named_parameters(),
    key=lambda named_parameter: named_parameter[1].numel(),
)
teacher_reference_before_step = teacher_reference_parameter.detach().clone()

# Reverse KL gives dense feedback over the vocabulary. Save the output-weight
# row for the first response token as a small, concrete student weight sample.
student_output_weight = student.get_output_embeddings().weight
student_update_token_id = response_token_ids[0, 0].item()
student_output_row_before_step = (
    student_output_weight[student_update_token_id].detach().clone()
)
pre_step_mean_reverse_kl = mean_response_reverse_kl.detach().clone()

print("Optimizer: AdamW")
print("Demonstration learning rate:", demonstration_learning_rate)
print("Student output row being tracked for token ID:", student_update_token_id)
print("Frozen teacher parameter being tracked:", teacher_reference_name)
print("Part 5 mean reverse KL before update:", pre_step_mean_reverse_kl.item())


## 2. Rebuild the loss and its gradients

Part 6 cleared its gradients after testing them, so we need one new forward-and-backward pass for the optimizer. We switch the student to training mode, reuse the fixed trajectory and detached teacher targets, and calculate the same response-only reverse KL.

We clip the total gradient norm to `1.0` before the update. Clipping rescales an unusually large gradient while preserving its overall direction. It is a stability measure, not the optimizer step itself.


In [ ]:
assert not globals().get("optimizer_step_has_run", False)
student.train()
student_optimizer.zero_grad(set_to_none=True)

# This is a fresh differentiable student pass over the same fixed token IDs.
optimizer_student_result = student(
    input_ids=training_complete_sequence_ids,
    attention_mask=training_complete_attention_mask,
    use_cache=False,
    return_dict=True,
)
optimizer_shifted_logits = optimizer_student_result.logits[:, :-1, :]
optimizer_response_logits = optimizer_shifted_logits[
    :, response_start_in_shifted_sequence:, :
]
assert optimizer_response_logits.shape == teacher_training_target_log_probs.shape

# Shape: [1, response_length, vocabulary_size]. Only the student side is
# connected to autograd; the detached teacher side remains a fixed target.
optimizer_student_log_probs = torch.log_softmax(
    optimizer_response_logits.float(), dim=-1
)
optimizer_student_probs = optimizer_student_log_probs.exp()
optimizer_response_reverse_kl = (
    optimizer_student_probs
    * (optimizer_student_log_probs - teacher_training_target_log_probs)
).sum(dim=-1)
optimizer_reverse_kl_loss = optimizer_response_reverse_kl.mean()
assert optimizer_response_reverse_kl.shape == response_token_ids.shape
assert optimizer_reverse_kl_loss.requires_grad
assert torch.isfinite(optimizer_reverse_kl_loss)

# backward() fills student gradients. clip_grad_norm_ returns the total norm
# before clipping and raises immediately if it is NaN or infinite.
optimizer_reverse_kl_loss.backward()
pre_clip_gradient_norm = torch.nn.utils.clip_grad_norm_(
    student.parameters(),
    max_norm=1.0,
    error_if_nonfinite=True,
)
assert torch.isfinite(pre_clip_gradient_norm)
assert pre_clip_gradient_norm.item() > 0
assert student_output_weight.grad is not None
assert torch.isfinite(student_output_weight.grad).all()
assert student_output_weight.grad[student_update_token_id].abs().max().item() > 0
assert all(parameter.grad is None for parameter in teacher.parameters())

print("Training loss before optimizer step:", optimizer_reverse_kl_loss.item())
print("Student gradient norm before clipping:", pre_clip_gradient_norm.item())
print("Gradient norm was clipped to at most: 1.0")


## 3. Apply exactly one optimizer step

`student_optimizer.step()` is the moment the student changes. AdamW uses the gradients to update the student parameters. Immediately afterward we set the guard, verify that the tracked student output-weight row changed, and verify that the tracked teacher parameter stayed exactly the same.

The optimizer state is then deleted to free GPU memory. The updated student remains in memory, but it is not written to disk or uploaded anywhere.


In [ ]:
# Keep this guard in the mutation cell itself: rerunning this cell must not
# silently apply a second update to the same now-stale trajectory.
assert not globals().get("optimizer_step_has_run", False), (
    "The one permitted optimizer step has already run."
)
student_optimizer.step()
optimizer_step_has_run = True

student_output_row_after_step = (
    student_output_weight[student_update_token_id].detach().clone()
)
student_output_row_max_change = (
    student_output_row_after_step.float()
    - student_output_row_before_step.float()
).abs().max()
assert student_output_row_max_change.item() > 0
assert torch.equal(
    teacher_reference_parameter.detach(),
    teacher_reference_before_step,
)
assert all(parameter.grad is None for parameter in teacher.parameters())

# The update is complete. Remove gradients and optimizer state before the
# post-step evaluation so they do not consume unnecessary GPU memory.
student_optimizer.zero_grad(set_to_none=True)
assert all(parameter.grad is None for parameter in student.parameters())
del student_optimizer
torch.cuda.empty_cache()

print("Optimizer steps applied: 1")
print("Maximum absolute change in tracked student row:", student_output_row_max_change.item())
print("Tracked teacher weight unchanged: True")


## 4. Measure the updated student without another update

We put the student back in evaluation mode and recalculate reverse KL on the same fixed response. This lets us observe the immediate effect of the step. It is a **training-example diagnostic**, not a fair evaluation: the optimizer just saw this exact trajectory.

A single AdamW step is not mathematically guaranteed to lower a finite-step loss for every learning rate, so the checks require a finite measured result and a real weight change rather than forcing the loss to decrease. General improvement must later be tested on separate prompts in the baseline evaluation notebook.


In [ ]:
assert globals().get("optimizer_step_has_run", False)
student.eval()

# This is measurement only. The updated student reads the same fixed
# prefixes so the before/after reverse-KL values are directly comparable.
with torch.no_grad():
    post_step_student_result = student(
        input_ids=training_complete_sequence_ids,
        attention_mask=training_complete_attention_mask,
        use_cache=False,
        return_dict=True,
    )
    post_step_response_logits = post_step_student_result.logits[
        :, :-1, :
    ][:, response_start_in_shifted_sequence:, :]
    post_step_student_log_probs = torch.log_softmax(
        post_step_response_logits.float(), dim=-1
    )
    post_step_student_probs = post_step_student_log_probs.exp()
    post_step_response_reverse_kl = (
        post_step_student_probs
        * (post_step_student_log_probs - teacher_training_target_log_probs)
    ).sum(dim=-1)
    post_step_mean_reverse_kl = post_step_response_reverse_kl.mean()

reverse_kl_change = post_step_mean_reverse_kl - pre_step_mean_reverse_kl
assert post_step_response_reverse_kl.shape == response_token_ids.shape
assert torch.isfinite(post_step_response_reverse_kl).all()
assert torch.isfinite(post_step_mean_reverse_kl)
assert student_output_row_max_change.item() > 0
assert torch.equal(
    teacher_reference_parameter.detach(),
    teacher_reference_before_step,
)
assert all(parameter.grad is None for parameter in student.parameters())
assert all(parameter.grad is None for parameter in teacher.parameters())

optimizer_step_results = pd.DataFrame({
    "measurement": ["before update", "after update", "change (after - before)"],
    "mean_reverse_kl_nats": [
        pre_step_mean_reverse_kl.item(),
        post_step_mean_reverse_kl.item(),
        reverse_kl_change.item(),
    ],
})
display(optimizer_step_results)

optimizer_step_verified = True
print("Student weight changed: True")
print("Teacher weight changed: False")
print("Mean reverse KL before update:", pre_step_mean_reverse_kl.item())
print("Mean reverse KL after update:", post_step_mean_reverse_kl.item())
print("Mean reverse KL change:", reverse_kl_change.item())
print("All single-optimizer-step tests passed.")


## What this part proved

If you see **All single-optimizer-step tests passed**:

- The student generated the trajectory before this update.
- The frozen teacher supplied full next-token target distributions on the student's prefixes.
- Reverse KL produced student gradients, and AdamW applied exactly one update.
- A tracked student weight changed, while the tracked teacher weight stayed identical.
- We measured the updated student's reverse KL on the same example without applying another update.

In simple terms, we have completed one miniature on-policy GKD iteration: **sample from the student → score with the teacher → compare distributions → update the student**. The updated model exists only in this runtime and has not been saved. One training example cannot show that the model became generally better.

This completes the purpose of `01_reverse_kl_demo.ipynb`. The next notebook should be `02_baseline_evaluation.ipynb`, where we build a small multi-prompt evaluation before constructing repeated training. That baseline will give us something honest to compare against after later distillation runs.


# Complete notebook recap: what we built and what it means

This section explains the entire notebook as one connected process. It is intentionally detailed and uses simple language. You should be able to return to this recap later without remembering every PyTorch operation above.

## The goal in one sentence

We built and checked one small **on-policy distillation iteration**: the small student wrote a response, the large frozen teacher evaluated the student's next-token choices, reverse KL measured the difference between their probability distributions, and one optimizer step used that difference to change only the student.

This notebook follows the central idea in [*On-Policy Distillation of Language Models: Learning from Self-Generated Mistakes*](https://arxiv.org/abs/2306.13649). The paper calls the broader method **Generalized Knowledge Distillation**, or **GKD**.

## The complete flow

```text
Prompt
  ↓
Student samples its own response
  ↓
Keep the exact prompt and response token IDs
  ↓
Student scores every possible next token at each response position
  ↓
Frozen teacher scores every possible next token at the same positions
  ↓
Reverse KL measures how different the two distributions are
  ↓
Backward computes student gradients
  ↓
AdamW applies one update to the student only
  ↓
The updated student must generate fresh responses for the next iteration
```

The phrase **on-policy** comes from the second step. The training response came from the student that we were about to update. It was not a prewritten answer and was not generated by the teacher.

## The main pieces and their roles

| Piece | What it means in this notebook |
|---|---|
| Prompt | The arithmetic question given to the student |
| Student | `Qwen/Qwen3-0.6B-Base`, the smaller model we want to improve |
| Teacher | `Qwen/Qwen3-32B`, the larger model that supplies target probabilities |
| Tokenizer | The rule that converts text pieces into integer token IDs |
| Trajectory | The exact sequence of response token IDs sampled from the student |
| Logits | Raw model scores for every possible next token |
| Log probabilities | Normalized scores written in logarithmic form |
| Reverse KL | A number describing the difference between the student's and teacher's full next-token distributions |
| Gradient | Information about how each student parameter would need to move to change the loss |
| Optimizer | The procedure that turns gradients into actual student-weight changes |

## Part 1: the student generated the data

We started with one arithmetic prompt. The student sampled a response with temperature `1.0` and a fixed random seed. Temperature `1.0` left the student's distribution unchanged, and sampling produced a trajectory from the student's current behavior. That is the first on-policy ingredient.

The saved A100 run produced:

- `36` prompt tokens
- `128` response tokens
- `164` total tokens

The response stopped at the `128`-token limit, so it was cut off before finishing its written solution. That is acceptable for this mechanics demonstration because the notebook studies token generation and scoring. It does not use this one response as evidence of model quality.

We preserved `complete_sequence_ids` instead of decoding the response and tokenizing it again. The original IDs are the exact actions sampled from the student. Re-tokenizing text could change boundaries or special tokens and would weaken the alignment test.

## Part 2: the student scored what it had generated

Generation tells us which tokens the student selected. It does not automatically give us a clean score for every selected token in the format needed later, so we passed the complete prompt-plus-response sequence through the student again.

The student's logits had shape:

```text
[batch size, sequence length, vocabulary size]
[1,          164,             151936]
```

At every sequence position, the model produced `151936` scores: one score for every vocabulary token that could come next.

A causal language model always predicts the **next** token. The logits at position `t` predict the token stored at position `t + 1`. We therefore removed the last logits position and the first target token. This is the causal-language-model shift.

The most important boundary in the notebook is:

```text
final prompt token                 first response token
full-sequence position 35          full-sequence position 36
          logits at 35  ─────────▶ token at 36
```

That is why response scores begin at `prompt_length - 1` in the shifted logits. Starting at `prompt_length` would drop the score for the first response token and move every later score out of alignment.

After shifting and slicing, we had exactly one student log probability for each of the `128` response tokens. Prompt-token scores were excluded.

## Part 3: we checked that both models speak the same token-ID language

A token ID is only meaningful relative to a tokenizer. If ID `123` represented one text piece for the student and a different text piece for the teacher, passing the student's IDs into the teacher would ask the teacher to read different text. Any comparison would then be invalid.

We compared the student and teacher tokenizer mappings and checked the exact prompt and response IDs. The saved run found:

- Student vocabulary capacity: `151936`
- Teacher vocabulary capacity: `151936`
- Student token-mapping mismatches: `0`
- Teacher-only tokenizer entries: `0`
- Prompt ID meaning mismatches: `0`
- Response ID meaning mismatches: `0`

This proved that the two Qwen3 checkpoints assign the same meanings to the token IDs used here. It allowed us to compare their full vocabulary distributions directly.

## Part 4: the frozen teacher scored the student's trajectory

We loaded the 32B teacher in BF16 on the A100. Its checkpoint was about `61.0 GiB`, which is why this part requires a high-memory GPU. The teacher was placed in evaluation mode, and every teacher parameter had `requires_grad=False`.

The teacher did not generate a replacement answer. It read the exact prompt and response prefixes produced by the student. At each response position, it answered this question:

> Given the prompt and all response tokens before this position, what probability would I assign to every possible next token?

We applied the same one-token shift and response boundary used for the student. The teacher response logits had shape `[1, 128, 151936]`, and the selected teacher token scores had shape `[1, 128]`. This gave one teacher log probability for every token the student actually generated, plus a full teacher distribution for every response position.

## Part 5: reverse KL compared the full distributions

A generated token's student score and teacher score are useful to inspect, but subtracting those two selected-token scores is not the KL divergence. KL compares **all possible next tokens** at a position.

For reverse KL, the student distribution supplies the weights:

```text
KL(student || teacher)
= sum over vocabulary of:
  student_probability
  × (student_log_probability - teacher_log_probability)
```

We calculated one reverse-KL value at each of the `128` response positions and averaged them. The saved run produced a mean reverse KL of approximately `0.1592` **nats**. A nat is the unit produced when logarithms use base `e`, which is what PyTorch's `log_softmax` uses.

A KL value of zero would mean the two next-token distributions are identical at that position. A larger value means they disagree more. This number measures distribution disagreement; it does not directly say whether the written answer is correct.

## Part 6: we proved that the loss reaches only the student

The Part 5 calculation used `torch.no_grad()` because it was a measurement. Training requires an autograd graph, so we ran the student again with gradient tracking enabled.

The original generated IDs came from `torch.inference_mode()`. PyTorch marks tensors created there as inference tensors, and an embedding operation cannot save such a tensor for backward. We fixed this by cloning the IDs outside inference mode. The clone contained the same integers and therefore represented the same trajectory, but it was safe for autograd.

We detached the teacher log probabilities. This made the intended learning boundary explicit:

```text
teacher distributions: fixed targets
student distributions: differentiable predictions
sampled token IDs: fixed trajectory data
```

Calling `backward()` produced gradients for all `310` participating student parameter tensors. All `310` had nonzero gradients, the total gradient L2 norm was about `18.55`, and the teacher received zero gradients. A saved student weight remained unchanged because `backward()` calculates gradients but does not update weights.

This is the paper's important stop-gradient idea in practical form: we do not differentiate through the random act of sampling token IDs. We differentiate the divergence calculated afterward on those fixed sampled prefixes.

## Part 7: one optimizer step completes the miniature iteration

Part 7 creates an AdamW optimizer containing only trainable student parameters. It rebuilds the loss and gradients, clips the total gradient norm to `1.0`, and calls `student_optimizer.step()` exactly once. That call is the moment the student weights change.

The notebook tracks one student output-weight row and one small teacher parameter. After a successful Part 7 run, it verifies that the student sample changed and that the teacher sample stayed bit-for-bit identical. It then deletes the optimizer state to release GPU memory and measures reverse KL again without applying another update.

The one-step guard is important. After the student changes, the existing trajectory came from the previous version of the student. Reusing it for repeated updates would gradually stop being on-policy. The guard prevents an accidental second optimizer step on that stale trajectory.

The before-and-after loss is measured on the same example the optimizer just saw. It is useful for checking the mechanics, but it is not an unbiased evaluation of model quality.

## What we have genuinely verified

For this one prompt and one sampled response, the notebook verifies that:

1. The response really came from the student with temperature `1.0`.
2. Prompt and response token boundaries were preserved.
3. Student next-token scores were aligned with the tokens they predicted.
4. Student and teacher token IDs had matching meanings.
5. The frozen teacher scored the exact same prefixes and response positions.
6. Reverse KL used the full vocabulary distributions, not only the sampled tokens.
7. Prompt positions were excluded from the response loss.
8. The sampled token IDs were treated as fixed data during backpropagation.
9. Gradients flowed into the student and did not flow into the teacher.
10. `backward()` alone did not update weights.
11. One guarded optimizer step changed the student while leaving the teacher frozen, once Part 7 reports success.

These checks turn an abstract paper equation into a concrete, inspectable PyTorch computation.

## What this notebook does not prove

This notebook does not yet demonstrate that the student became generally better. It uses one arithmetic prompt, one sampled response, and at most one update. The response was also truncated at the generation limit.

We have not yet:

- Evaluated the original student on a collection of held-out prompts
- Defined accuracy or another task-quality metric across a dataset
- Compared an updated student with a reproducible baseline
- Repeated the generate-score-update cycle for many batches
- Generated a fresh trajectory after each student update
- Saved an updated student checkpoint
- Added LoRA or another parameter-efficient training method
- Tuned the learning rate, batch size, divergence, or generation settings
- Tested whether reverse KL is the best divergence for this task

Those are experiment-design questions rather than token-alignment questions. Separating them is useful: we now understand and trust the core mechanics before scaling them up.

## Why the updated student needs fresh trajectories

Suppose the original student is called `student_0`. It generated this notebook's response. After an optimizer step, we have `student_1`. The saved response was on-policy for `student_0`, because `student_0` generated it. It is historical data for `student_1`.

A repeated on-policy loop therefore looks like this:

```text
student_0 generates response_0 → update → student_1
student_1 generates response_1 → update → student_2
student_2 generates response_2 → update → student_3
...
```

The teacher remains frozen throughout. Its role is to provide a stable target while the student's behavior and training data evolve together.

## Why the next notebook is baseline evaluation

Before running many training updates, we need to measure the original student. Otherwise, a later score has nothing reliable to compare against. `02_baseline_evaluation.ipynb` should run the unchanged student on a small set of prompts using a fixed evaluation procedure and store the results.

The remaining project sequence is:

1. **`02_baseline_evaluation.ipynb`** — measure the original student on multiple prompts.
2. **`03_sft_cold_start.ipynb`** — prepare or study the supervised fine-tuning starting point used by the paper. The paper assumes the student already has enough ability to generate useful trajectories.
3. **`04_on_policy_distillation.ipynb`** — build the repeated batched loop: generate fresh student responses, obtain frozen-teacher distributions, calculate the chosen GKD divergence, update the student, and evaluate periodically.

## A simple mental model to keep

Think of the student as someone practicing a problem aloud and the teacher as a coach who watches that exact attempt. The coach does not rewrite the attempt. At every word, the coach shows which next choices it considered more or less plausible. Reverse KL turns the difference between the student's choices and the coach's choices into a learning signal. Backward calculates how the student should change, and the optimizer applies that change. The student then needs to attempt a new problem with its updated behavior.

That cycle is the heart of this project. This notebook has reduced it to one example and checked each boundary carefully.
